### YOLOv11n Transfer Learning from Fashionpedia Dataset

In [1]:
import os
import tqdm
import numpy as np
from datasets import load_dataset
from ultralytics import YOLO

In [2]:
#Download Dataset
ds = load_dataset("detection-datasets/fashionpedia")

#### Convert COCO to YOLO

In [7]:
# Create directory structure
output_dir = "/home/tommytang111/Projects/Drone/data/yolo_format"
os.makedirs(f"{output_dir}/images/train", exist_ok=True)
os.makedirs(f"{output_dir}/images/val", exist_ok=True)  
os.makedirs(f"{output_dir}/labels/train", exist_ok=True)
os.makedirs(f"{output_dir}/labels/val", exist_ok=True)

In [4]:
# Get class names
class_names = ds["train"].features["objects"].feature["category"].names
num_classes = len(class_names)
print(f"Found {num_classes} classes in Fashionpedia dataset")

Found 46 classes in Fashionpedia dataset


In [5]:
def coco_to_yolo_bbox(bbox, img_width, img_height):
    """Convert COCO format [x_min, y_min, width, height] to YOLO format [x_center, y_center, width, height] (normalized)"""
    x_min, y_min, width, height = bbox
    
    # Handle edge cases with invalid bounding boxes
    if width <= 0 or height <= 0:
        return None
        
    # Convert to YOLO format (normalized)
    x_center = (x_min + width / 2) / img_width
    y_center = (y_min + height / 2) / img_height
    width = width / img_width
    height = height / img_height
    
    # Ensure values are in valid range [0, 1]
    if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
        return None
        
    return [x_center, y_center, width, height]

In [26]:
# Process each split
for split in ["train", "val"]:
    yolo_split = "train" if split == "train" else "val"
    print(f"Processing {split} split...")
    
    for i, item in enumerate(tqdm.tqdm(ds[split])):
        # Get image
        img = item["image"]
        img_width, img_height = img.size
        
        # Create unique filename based on index
        filename = f"{i:06d}"
        
        # Save image
        img_path = f"{output_dir}/images/{yolo_split}/{filename}.jpg"
        img.save(img_path)
        
        # Save YOLO label
        label_path = f"{output_dir}/labels/{yolo_split}/{filename}.txt"
        
        with open(label_path, "w") as f:
            # Process each object
            for j in range(len(item["objects"]["bbox"])):
                # Get class ID and bounding box
                class_id = item["objects"]["category"][j]
                bbox = item["objects"]["bbox"][j]
                
                # Convert to YOLO format
                yolo_bbox = coco_to_yolo_bbox(bbox, img_width, img_height)
                
                # Skip invalid bounding boxes
                if yolo_bbox is None:
                    continue
                
                # Write to file: class_id x_center y_center width height
                bbox_str = " ".join([f"{coord:.6f}" for coord in yolo_bbox])
                f.write(f"{class_id} {bbox_str}\n")

Processing train split...


100%|██████████| 45623/45623 [03:42<00:00, 205.43it/s]


Processing val split...


100%|██████████| 1158/1158 [00:06<00:00, 187.36it/s]


In [27]:
# Create data.yaml file
yaml_content = f"""
train: {output_dir}/images/train
val: {output_dir}/images/val

nc: {num_classes}
names: {list(class_names)}
"""

with open(f"{output_dir}/data.yaml", "w") as f:
    f.write(yaml_content)

print(f"Conversion complete. Dataset saved to {output_dir}")
print(f"Created data.yaml with {num_classes} classes")

Conversion complete. Dataset saved to /home/tommytang111/Projects/Drone2/data/yolo_format
Created data.yaml with 46 classes


In [8]:
# Examine dataset structure and verify conversion success
!find {output_dir} -type f | wc -l
print("Sample label file content:")
!head -n 3 {output_dir}/labels/train/000000.txt

# Check class distribution
import glob
import re

def count_classes(label_dir):
    class_counts = [0] * num_classes
    for label_file in glob.glob(f"{label_dir}/*.txt"):
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                class_counts[class_id] += 1
    return class_counts

train_class_counts = count_classes(f"{output_dir}/labels/train")
val_class_counts = count_classes(f"{output_dir}/labels/val")

# Display top 10 classes
top_classes = sorted(range(len(train_class_counts)), 
                    key=lambda i: train_class_counts[i], 
                    reverse=True)[:10]

print("\nTop 10 classes by frequency:")
for i, class_id in enumerate(top_classes):
    print(f"{i+1}. {class_names[class_id]}: {train_class_counts[class_id]} train, {val_class_counts[class_id]} val")

93565
Sample label file content:
33 0.719941 0.447266 0.565982 0.343750
10 0.636364 0.600098 0.656891 0.649414

Top 10 classes by frequency:
1. sleeve: 45086 train, 1211 val
2. neckline: 33571 train, 894 val
3. pocket: 19116 train, 388 val
4. dress: 18478 train, 495 val
5. top, t-shirt, sweatshirt: 16083 train, 453 val
6. collar: 9978 train, 215 val
7. jacket: 7694 train, 177 val
8. pants: 7266 train, 218 val
9. zipper: 6520 train, 152 val
10. shirt, blouse: 6056 train, 102 val


#### Training

In [ ]:
#Load Model
model = YOLO('yolo11m')

100%|██████████| 5.35M/5.35M [00:00<00:00, 29.1MB/s]


In [19]:
model.model

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

In [22]:
# Optimized Transfer Learning for Highest IoU Results
pytorch_model = model.model

# For highest IoU, freeze backbone + early neck features
freeze_until_layer = 8  # Optimal for IoU performance

frozen_layers = []
trainable_layers = []

for i, (name, param) in enumerate(pytorch_model.named_parameters()):
    layer_num = int(name.split('.')[1]) if 'model.' in name and name.split('.')[1].isdigit() else -1
    
    if layer_num <= freeze_until_layer:
        param.requires_grad = False
        frozen_layers.append(name)
    else:
        param.requires_grad = True
        trainable_layers.append(name)

print(f"Frozen layers (0-{freeze_until_layer}): {len(frozen_layers)} parameters")
print(f"Trainable layers ({freeze_until_layer+1}-23): {len(trainable_layers)} parameters")

# Verify the freeze configuration
frozen_params = sum(1 for param in pytorch_model.parameters() if not param.requires_grad)
total_params = sum(1 for param in pytorch_model.parameters())
trainable_params = total_params - frozen_params

print(f"\nParameter Summary:")
print(f"Total parameters: {total_params}")
print(f"Frozen parameters: {frozen_params}")
print(f"Trainable parameters: {trainable_params}")
print(f"Percentage trainable: {trainable_params/total_params*100:.1f}%")

print(f"\nFrozen Layers (0-8): Robust feature extraction")
print("- Layers 0-6: Backbone (edges, textures, basic patterns)")
print("- Layers 7-8: Early neck (complex patterns, initial fusion)")

print(f"\nTrainable Layers (9-23): IoU optimization")
print("- Layer 9: SPPF (spatial pyramid pooling)")
print("- Layer 10: C2PSA (attention mechanism)")
print("- Layers 11-16: FPN (multi-scale feature fusion)")
print("- Layers 17-23: PAN + Detection head (precise localization)")

Frozen layers (0-8): 93 parameters
Trainable layers (9-23): 163 parameters

Parameter Summary:
Total parameters: 256
Frozen parameters: 93
Trainable parameters: 163
Percentage trainable: 63.7%

Frozen Layers (0-8): Robust feature extraction
- Layers 0-6: Backbone (edges, textures, basic patterns)
- Layers 7-8: Early neck (complex patterns, initial fusion)

Trainable Layers (9-23): IoU optimization
- Layer 9: SPPF (spatial pyramid pooling)
- Layer 10: C2PSA (attention mechanism)
- Layers 11-16: FPN (multi-scale feature fusion)
- Layers 17-23: PAN + Detection head (precise localization)


In [23]:
# Train YOLOv11m on Fashionpedia
results = model.train(
    data=f'{output_dir}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,  # Reduced batch size to start with, increase if GPU memory allows
    device=0,
    pretrained=True,
    patience=10,
    lr0=0.002, 
    warmup_epochs=3,
    optimizer='AdamW',
    name='yolov11m-fashionpedia'
)

print(f"Training complete. Best model saved at: {results.best}")

New https://pypi.org/project/ultralytics/8.3.147 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/home/tommytang111/Projects/Drone/data/yolo_format/data.yaml, epochs=100, time=None, patience=10, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolov11m-fashionpedia, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes

train: Scanning /home/tommytang111/Projects/Drone/data/yolo_format/labels/train... 45623 images, 206 backgrounds, 0 corrupt: 100%|██████████| 45623/45623 [00:35<00:00, 1269.71it/s]


train: New cache created: /home/tommytang111/Projects/Drone/data/yolo_format/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/home/tommytang111/.venvs/tf_gpu_env/lib/python3.11/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
val: Scanning /home/tommytang111/Projects/Drone/data/yolo_format/labels/val... 1158 images, 14 backgrounds, 0 corrupt: 100%|██████████| 1158/1158 [00:01<00:00, 696.69it/s]


val: New cache created: /home/tommytang111/Projects/Drone/data/yolo_format/labels/val.cache
Plotting labels to runs/detect/yolov11m-fashionpedia/labels.jpg... 
optimizer: AdamW(lr=0.002, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/yolov11m-fashionpedia
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      6.18G      1.679       3.33      2.122        324        640:  28%|██▊       | 401/1426 [01:33<03:59,  4.28it/s]


KeyboardInterrupt: 